# RSNA 2026 Knee MRI Abnormality Detection: submission notebook (gold-only head)

Part 3 of 3. This is the notebook built for the competition's code-only scoring run (no internet, 9-hour limit). It predicts the 12 findings for every test study with **image-only** features, because test studies come without radiology reports.

**How it works**
1. Load a small private *bundle* (a Kaggle dataset): the 58 gold studies' pooled features and labels, the ImageNet ResNet18 backbone weights (no download needed offline), and a `config.json` that records the preprocessing settings.
2. Fit one logistic regression per finding on the 58 gold studies (`C=0.005`, balanced class weights).
3. For each test study, pick one series per plane (fluid-sensitive preferred), run the frozen backbone on the central 60% of slices, **max-pool across slices**, and concatenate the three planes (3 x 512 = 1536 features).
4. Predict probabilities and write `submission.csv` in the sample submission's row order.

The `assert` statements are deliberate: this notebook fails loudly if the bundle's preprocessing settings, the submission schema, or the test metadata differ from what the features were trained with.

Cross-validated macro AUC for this configuration on the 58 gold studies is about **0.669** (see Part 2); it does not use the report-derived weak labels.

Outputs are cleared; the bundle and competition data are not part of this repository.

*Change from the submitted version: the file search in the setup cell is depth-limited (see the comment there).*

## 1. Setup: locate the competition data and the bundle, check the preprocessing config

In [ ]:
from pathlib import Path
import json
import os
import time
import numpy as np
import pandas as pd
import cv2
import pydicom
import torch
import torch.nn as nn
import torchvision.models as models
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from tqdm.auto import tqdm


DATA_ROOT_OVERRIDE = None
BUNDLE_DIR_OVERRIDE = None
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working')
OUTPUT_ROOT.mkdir(exist_ok=True)
OUTPUT_FILE = OUTPUT_ROOT / 'submission.csv'
# Avoid retaining an old submission after a failed rerun.
if OUTPUT_FILE.exists():
    OUTPUT_FILE.unlink()

def unique_file(name, max_depth=4):
    # Look at most `max_depth` directory levels down. The original recursive rglob walked every
    # DICOM file under /kaggle/input and took ~24 minutes on its own.
    found = []
    for depth in range(max_depth):
        found += sorted(INPUT_ROOT.glob('/'.join(['*'] * depth + [name])))
    if len(found) != 1:
        raise RuntimeError(f'Expected one {name}; found {found}. Set the path override.')
    return found[0]

DATA_ROOT = (Path(DATA_ROOT_OVERRIDE) if DATA_ROOT_OVERRIDE
             else unique_file('test_series.csv').parent)
BUNDLE_DIR = (Path(BUNDLE_DIR_OVERRIDE) if BUNDLE_DIR_OVERRIDE
              else unique_file('gold_training.npz').parent)
for name in ['gold_training.npz', 'resnet18_backbone.pth', 'config.json']:
    assert (BUNDLE_DIR / name).is_file(), f'Missing artifact: {name}'
config = json.loads((BUNDLE_DIR / 'config.json').read_text())
expected = dict(version=1, planes=['Axial', 'Coronal', 'Sagittal'],
                preferred_fluid=1, target_size=256, percentiles=[1, 99],
                central=[0.2, 0.8], slice_order='lexicographic_filename',
                pooling='max', C=0.005)
assert config == expected, 'Bundle preprocessing does not match this notebook'
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
PLANES = config['planes']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(min(4, os.cpu_count() or 1))
cv2.setNumThreads(0)
print('Competition:', DATA_ROOT, '| Bundle:', BUNDLE_DIR, '| Device:', device)
print('Versions:', torch.__version__, pydicom.__version__, cv2.__version__)


## 2. Fit the gold-only heads and load the offline backbone

In [ ]:
# Fit on ALL gold studies for final inference
with np.load(BUNDLE_DIR / 'gold_training.npz', allow_pickle=False) as d:
    F_gold = d['features'].copy()
    y = d['labels'].copy()
    gold_ids = d['ids'].astype(str)
    assert d['label_names'].tolist() == LABELS
assert F_gold.shape == (len(gold_ids), 1536)
assert y.shape == (len(gold_ids), 12)
assert len(set(gold_ids)) == len(gold_ids)
assert np.isfinite(F_gold).all() and np.isin(y, [0, 1]).all()
scaler = StandardScaler().fit(F_gold)
Xg = scaler.transform(F_gold)
heads = []
for j, label in enumerate(LABELS):
    assert len(np.unique(y[:, j])) == 2, f'Only one gold class for {label}'
    clf = LogisticRegression(C=config['C'], max_iter=2000, class_weight='balanced')
    clf.fit(Xg, y[:, j])
    assert clf.n_iter_.max() < clf.max_iter, f'No convergence for {label}'
    heads.append(clf)
print(f'Trained {len(heads)} heads on {len(y)} gold studies.')

# No weight download: the exported backbone has no classification FC parameters.
backbone = models.resnet18(weights=None)
backbone.fc = nn.Identity()
state = torch.load(BUNDLE_DIR / 'resnet18_backbone.pth', map_location='cpu', weights_only=True)
backbone.load_state_dict(state, strict=True)
backbone = backbone.to(device).eval()
mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


## 3. Feature extraction (must match the training-time extraction exactly)
Same resize, percentile normalization, slice range and series choice as the code that produced the stored gold features.

In [ ]:
def load_resize_normalize(path):
    ds = pydicom.dcmread(path)
    image = ds.pixel_array.astype(np.float32)
    assert image.ndim == 2, f'Expected one 2D image: {path}'
    image = cv2.resize(image, (256, 256), interpolation=cv2.INTER_AREA)
    lo, hi = np.percentile(image, [1, 99])
    if hi - lo < 1e-6:
        return np.zeros_like(image, dtype=np.float32)
    return ((np.clip(image, lo, hi) - lo) / (hi - lo)).astype(np.float32)

def central_files(series_dir):
    # Identical ordering and range to the original feature extraction.
    files = sorted(os.listdir(series_dir))
    if not files:
        raise RuntimeError(f'Empty series: {series_dir}')
    lo = int(len(files) * 0.2)
    hi = max(int(len(files) * 0.8), lo + 1)
    return [series_dir / f for f in files[lo:hi]]

def select_series(study_rows, plane):
    rows = study_rows.loc[study_rows['Anatomical_Plane'] == plane]
    if rows.empty:
        return None
    preferred = rows.loc[rows['Fluid_Sensitive'] == 1]
    chosen = preferred if not preferred.empty else rows.loc[rows['Fluid_Sensitive'] != 1]
    return str(chosen.iloc[0]['SeriesInstanceUID']) if not chosen.empty else None

@torch.inference_mode()
def series_feature(series_dir, batch_size=32):
    paths = central_files(series_dir)
    features = []
    for start in range(0, len(paths), batch_size):
        images = np.stack([load_resize_normalize(p) for p in paths[start:start + batch_size]])
        xb = torch.from_numpy(images).unsqueeze(1).repeat(1, 3, 1, 1)
        features.append(backbone(((xb - mean) / std).to(device)).cpu().numpy())
    result = np.concatenate(features, axis=0).max(axis=0)
    assert result.shape == (512,) and np.isfinite(result).all()
    return result


## 4. Read the test metadata and smoke-test the DICOM decoder
Test study IDs come from `sample_submission.csv` (which fixes the output row order). Decoding one series up front makes a missing offline decoder fail early instead of after a long run.

In [ ]:
# Discover IDs from the scoring run's sample submission, preserving its row order.
sample = pd.read_csv(DATA_ROOT / 'sample_submission.csv', dtype={'StudyInstanceUID': str})
test_series = pd.read_csv(DATA_ROOT / 'test_series.csv',
                          dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
assert list(sample.columns) == ['StudyInstanceUID'] + LABELS, 'Unexpected submission schema'
assert len(sample) > 0 and sample['StudyInstanceUID'].notna().all()
assert sample['StudyInstanceUID'].is_unique
required = {'StudyInstanceUID', 'SeriesInstanceUID', 'Anatomical_Plane', 'Fluid_Sensitive'}
assert required.issubset(test_series.columns)
assert test_series[list(required)].notna().all().all()
test_series['Fluid_Sensitive'] = pd.to_numeric(test_series['Fluid_Sensitive'], errors='raise')
assert test_series['Fluid_Sensitive'].isin([0, 1]).all()
assert test_series['Anatomical_Plane'].isin(PLANES).all()
groups = {sid: rows for sid, rows in test_series.groupby('StudyInstanceUID', sort=False)}
test_ids = sample['StudyInstanceUID'].tolist()
assert set(test_ids).issubset(groups), 'Test studies missing from test_series.csv'
print(f'Test studies: {len(test_ids)}; test series: {len(test_series)}')

# Decode and run one series now, so missing offline decoders fail before a long run.
first = test_series.loc[test_series['StudyInstanceUID'].isin(test_ids)].iloc[0]
first_dir = DATA_ROOT / 'test_series' / first['StudyInstanceUID'] / first['SeriesInstanceUID']
probe = series_feature(first_dir)
print('Offline DICOM + backbone smoke test passed:', probe.shape)


## 5. Extract features for every test study

In [ ]:
started = time.time()
test_features = np.zeros((len(test_ids), 3, 512), dtype=np.float32)
missing_planes = []
for n, sid in enumerate(tqdm(test_ids, desc='Test studies')):
    rows = groups[sid]
    for p, plane in enumerate(PLANES):
        series_id = select_series(rows, plane)
        if series_id is None:
            # Same zero-feature fallback as the training extractor for absent planes.
            missing_planes.append((sid, plane))
            continue
        series_dir = DATA_ROOT / 'test_series' / sid / series_id
        try:
            test_features[n, p] = series_feature(series_dir)
        except Exception as exc:
            raise RuntimeError(f'Extraction failed for study {sid}, plane {plane}, series {series_id}. '
                               'Check the attached DICOM decoder packages and data.') from exc
    if n == 0 or (n + 1) % 100 == 0:
        elapsed = time.time() - started
        remaining = elapsed / (n + 1) * (len(test_ids) - n - 1)
        print(f'{n+1}/{len(test_ids)} | elapsed {elapsed/60:.1f} min | estimated remaining {remaining/60:.1f} min')
F_test = test_features.reshape(len(test_ids), 1536)
assert np.isfinite(F_test).all()
print('Missing planes filled with zeros:', len(missing_planes))
print(f'Extraction took {(time.time() - started)/60:.1f} min')


## 6. Predict and write `submission.csv`

In [ ]:
Xt = scaler.transform(F_test)
predictions = np.column_stack([head.predict_proba(Xt)[:, 1] for head in heads])
assert predictions.shape == (len(sample), 12)
assert np.isfinite(predictions).all()
assert ((predictions >= 0) & (predictions <= 1)).all()
submission = sample.copy()
submission[LABELS] = predictions
assert submission['StudyInstanceUID'].tolist() == test_ids
assert list(submission.columns) == ['StudyInstanceUID'] + LABELS
submission.to_csv(OUTPUT_FILE, index=False)
check = pd.read_csv(OUTPUT_FILE, dtype={'StudyInstanceUID': str})
assert check.shape == submission.shape and not check.isna().any().any()
assert check['StudyInstanceUID'].tolist() == test_ids
print(f'Saved and checked {OUTPUT_FILE}: {check.shape}')
display(check.head())
